In [ ]:
!pip install peft==0.11.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
# Настройка промпта для Jungle

prompt_config = {
    "task": "Predict the endgame result of Jungle Chess (Dou Shou Qi)",
    "labels": ["white_win", "draw", "black_win"],
    "entity": "Game Position",
    "question": "Based on the rank, file, and strength of the white and black pieces, what is the game result? (White wins, Black wins, or Draw)"
}

openml_id = 41027

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464, prompt_config=None):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])
        class_names = prompt_config['labels']
    else:
        class_names = sorted(df[target_name].unique().tolist())

    return df, feature_names, target_name, class_names

def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):

    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name, class_names = load_dataset(openml_id, prompt_config)
train_df, val_df, test_df = split_dataset(df, target_name)

In [ ]:
df[target_name].value_counts()

,count
class,
2,23062
0,17422
1,4335


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44819 entries, 0 to 44818
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   white_piece0_strength  44819 non-null  int64
 1   white_piece0_file      44819 non-null  int64
 2   white_piece0_rank      44819 non-null  int64
 3   black_piece0_strength  44819 non-null  int64
 4   black_piece0_file      44819 non-null  int64
 5   black_piece0_rank      44819 non-null  int64
 6   class                  44819 non-null  int64
dtypes: int64(7)
memory usage: 2.4 MB


In [ ]:
for name, df in [("train", train_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")

    for i, label in enumerate(prompt_config['labels']):
        count = counts.get(i, 0)
        pct = pcts.get(i, 0.0)
        print(f"  {label:10s}: {count:5d} — {pct:.1f}%")


train (всего: 26891):
  white_win : 10453 — 38.9%
  draw      :  2601 — 9.7%
  black_win : 13837 — 51.5%

test (всего: 8964):
  white_win :  3484 — 38.9%
  draw      :   867 — 9.7%
  black_win :  4613 — 51.5%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['labels'][int(row[target_name])]
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

The value of white_piece0_strength is 6. The value of white_piece0_file is 6. The value of white_piece0_rank is 1. The value of black_piece0_strength is 5. The value of black_piece0_file is 2. The value of black_piece0_rank is 1.: class -> white_win
The value of white_piece0_strength is 6. The value of white_piece0_file is 6. The value of white_piece0_rank is 1. The value of black_piece0_strength is 5. The value of black_piece0_file is 2. The value of black_piece0_rank is 1.


,white_piece0_strength,white_piece0_file,white_piece0_rank,black_piece0_strength,black_piece0_file,black_piece0_rank,class
33644,6,6,1,5,2,1,0


In [ ]:
def parse_prediction(response, prompt_config):
    """Парсинг ответа модели в номер класса"""
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    # Проверка каждого класса
    for i, class_name in enumerate(prompt_config['labels']):
        class_lower = class_name.lower()

        # Точное совпадение
        if response == class_lower:
            return i

        # Начинается с имени класса
        if response.startswith(class_lower):
            return i

        # Содержит как отдельное слово
        if class_lower in response.split():
            return i

    # Не удалось распознать - возвращаем первый класс
    print(f"Warning: Could not parse '{response}' (expected one of {prompt_config['labels']})")
    return 0

response = "good"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "no"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'good'
Prediction: 0

Response: 'no'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    pr = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

    if y_prob is not None:
        try:
            roc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
        except:
            roc = 0.0
    else:
        roc = 0.0
    return roc, f1, acc, pr, rec

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    """Bootstrap метрики с доверительными интервалами"""
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            acc = accuracy_score(y_true_boot, y_pred_boot)
            f1 = f1_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)

            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
            else:
                auc = 0.0

            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    labels_str = "', '".join(prompt_config['labels'])

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"Answer with only one word from: '{labels_str}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['labels'][int(ex[target_name])]

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            attention_mask=model_inputs.attention_mask,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для всех классов
    first_token_logits = outputs.scores[0][0]

    labels_ids = []
    for labels_name in prompt_config['labels']:
        labels_id = tokenizer.encode(labels_name, add_special_tokens=False)[0]
        labels_ids.append(labels_id)

    labels_logits = torch.stack([first_token_logits[cid] for cid in labels_ids])
    probs = F.softmax(labels_logits, dim=0)

    # Возвращаем ответ и вероятности всех классов
    return response, probs.cpu().numpy()

# Тест
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, probs = predict_single_with_prob(test_prompt, prompt_config, model, tokenizer, device)
print(f"Response: '{response}'")
print(f"Probabilities: {dict(zip(prompt_config['labels'], probs))}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Response: 'draw'
Probabilities: {'white_win': np.float32(2.1024329e-07), 'draw': np.float32(0.9999995), 'black_win': np.float32(2.3823685e-07)}


# 3.1 Zero-shot

In [ ]:
print("Zero-shot классификация")

seed = 42

# Запускаем zero-shot предсказания
y_true_zero = []
y_pred_zero = []
y_prob_zero = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    response, prob = predict_single_with_prob(prompt, prompt_config, model, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_zero.append(row[target_name])
    y_pred_zero.append(prediction)
    y_prob_zero.append(prob)

zero_shot_time = time.time() - start_time

print(f"zero_shot_time: {zero_shot_time}")
print(f"zero_shot_time / len(y_true_zero) {zero_shot_time / len(y_true_zero)}")

Zero-shot классификация


  0%|          | 0/8964 [00:00<?, ?it/s]

zero_shot_time: 1162.8173189163208
zero_shot_time / len(y_true_zero) 0.12972080755425267


In [ ]:
# Вычисляем метрики
roc_zero, f1_zero, acc_zero, pr_zero, rec_zero = compute_metrics(np.array(y_true_zero), np.array(y_pred_zero), np.array(y_prob_zero))

print("Результаты zero-shot:")
print(f"ROC AUC: {roc_zero}")
print(f"F1 Score: {f1_zero}")
print(f"Accuracy: {acc_zero}")
print(f"Precision: {pr_zero}")
print(f"Recall: {rec_zero}")

Результаты zero-shot:
ROC AUC: 0.3920809718958828
F1 Score: 0.15228806068356157
Accuracy: 0.15082552431950022
Precision: 0.2095717368427311
Recall: 0.25848219441181064


In [ ]:
zero_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_zero),
    np.array(y_pred_zero),
    np.array(y_prob_zero),
    n_iter=1000
)

print("\nРезультаты zero-shot (bootstrap метрики с доверительными интервалами):")
for key, value in zero_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты zero-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.3922±0.0037
  F1: 0.1524±0.0038
  Accuracy: 0.1510±0.0038
  Precision: 0.2097±0.0057
  Recall: 0.2589±0.0063


ROC-AUC (0.39): Крайне низкий результат, значительно хуже случайного классификатора.

Это подтверждает, что без специальных знаний о правилах игры "Jungle Chess" модель совершенно не способна интерпретировать координаты и типы фигур в табличном виде.

В отличие от медицинских данных, где названия признаков помогают модели, здесь абстрактные игровые состояния оказались для LLM в режиме zero-shot нечитаемыми.

Accuracy (0.15): Модель практически не находит верных ответов, что делает данный датасет самым сложным для анализа в исходном состоянии.

Время тестирования: ~ 20 мин.

# 3.2 Few-shot

In [ ]:
# Выбираем 64 примера из train для контекста
n_examples = 64

seed = 42

num_classes = len(prompt_config['labels'])
n_per_class = n_examples // num_classes

sampled_dfs = []

for cls_idx in range(num_classes):
    cls_examples = train_df[train_df[target_name] == cls_idx].sample(n=n_per_class, random_state=seed)
    sampled_dfs.append(cls_examples)

few_shot_df = pd.concat(sampled_dfs).sample(frac=1, random_state=seed).reset_index(drop=True)

print(f"\nВыбрано {len(few_shot_df)} примеров для контекста:")
for cls_idx, label in enumerate(prompt_config['labels']):
    count = (few_shot_df[target_name] == cls_idx).sum()
    print(f"  {label}: {count}")

few_shot_examples = [row for _, row in few_shot_df.iterrows()]

# Few-shot предсказания
y_true_few = []
y_pred_few = []
y_prob_few = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples)
    response, prob = predict_single_with_prob(prompt, prompt_config, model, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_few.append(row[target_name])
    y_pred_few.append(prediction)
    y_prob_few.append(prob)

few_shot_time = time.time() - start_time

print(f"few_shot_time: {few_shot_time}")
print(f"few_shot_time / len(y_true_few) {few_shot_time / len(y_true_few)}")


Выбрано 63 примеров для контекста:
  white_win: 21
  draw: 21
  black_win: 21


  0%|          | 0/8964 [00:00<?, ?it/s]

few_shot_time: 5154.423163175583
few_shot_time / len(y_true_few) 0.5750137397563122


In [ ]:
roc_few, f1_few, acc_few, pr_few, rec_few = compute_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few)
)
print("Результаты few-shot:")
print(f"ROC AUC: {roc_few}")
print(f"F1 Score: {f1_few}")
print(f"Accuracy: {acc_few}")
print(f"Precision: {pr_few}")
print(f"Recall: {rec_few}")


Результаты few-shot:
ROC AUC: 0.4329718510621663
F1 Score: 0.22067229410827596
Accuracy: 0.25937081659973227
Precision: 0.23227748161744355
Recall: 0.22925904080550588


In [ ]:
few_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few),
    n_iter=1000
)

print("\nРезультаты few-shot (bootstrap метрики с доверительными интервалами):")
for key, value in few_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты few-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.4331±0.0043
  F1: 0.2208±0.0042
  Accuracy: 0.2595±0.0046
  Precision: 0.2324±0.0042
  Recall: 0.2294±0.0050


In [ ]:
from google.colab import runtime
runtime.unassign()

ROC-AUC (0.43): Наблюдается заметный прогресс (+4%), однако результат всё еще не достигает уровня случайного угадывания (0.50).

Хотя несколько примеров помогли модели слегка улучшить метрики, этого явно недостаточно для понимания сложной игровой логики.

Accuracy: Точность выросла почти в два раза (с 15% до 26%), что указывает на попытки модели подражать паттернам из примеров контекста.

Вывод: Датасет Jungle является наиболее "неинтуитивным" для языковых моделей.

Для достижения приемлемых результатов здесь требуется полноценное дообучение (Fine-tuning), так как простых подсказок в контексте недостаточно для выстраивания логических цепочек игровых стратегий.

Время выполнения: ~ 1 ч 30 мин

Использование оперативная памяти графического процесса: 10.1/80GB

Графический процессор A100